# Assignment: From Translator to Text Generator

## Converting our NMT Transformer into a Character-Level GPT

In this assignment, you will take the encoder-decoder Transformer we built for
Spanish-English translation and **simplify it** into a decoder-only model (GPT-style)
that generates Shakespeare one character at a time.

This is mostly a task of *removing things* and *rewiring what's left*.
You are not building anything fundamentally new — you're taking an architecture
you already understand and stripping it down to its core.

### The key insight

Our NMT Transformer has two jobs:
1. **Understand** the source sentence (encoder with self-attention)
2. **Generate** the target sentence (decoder with masked self-attention + cross-attention)

A GPT only does job #2 — it generates text. There's no source sentence,
so there's no encoder and no cross-attention. What's left is a stack of
masked self-attention + feed-forward layers. That's it.

### What you'll do

1. Remove the encoder
2. Remove cross-attention from the decoder layer
3. Adapt the embeddings for character-level input
4. Modify the forward pass
5. Train on Shakespeare and generate text

### What's provided for you

- The character tokenizer and data loading (Sections 1-2)
- Components copied directly from the NMT notebook (Section 3)
- The text generation function (Section 7)
- The training loop (Section 8)
- Shape-checking test cells after each section you write

### What you write

- The modified decoder layer (Section 4)
- The CharGPT model class (Section 5)
- The forward method (Section 6)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt
import time
import os

---
## Section 1: Data — Shakespeare Corpus (PROVIDED)

Same dataset as our char-RNN notebook. Nothing to modify here.

In [3]:
import urllib.request

url = "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
filepath = "shakespeare.txt"

if not os.path.exists(filepath):
    print("Downloading Shakespeare...")
    urllib.request.urlretrieve(url, filepath)

with open(filepath, 'r') as f:
    text = f.read()

print(f"Total characters: {len(text):,}")
print(f"First 200 characters:\n{text[:200]}")

Total characters: 1,115,394
First 200 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


---
## Section 2: Character Tokenizer and Batching (PROVIDED)

Our NMT model used word-level vocabularies with ~24,000 tokens.
Here the vocabulary is just the ~65 unique characters in Shakespeare.

Instead of discrete sentence pairs, we sample random windows from one
long character stream.

**Question before you continue:** In our NMT notebook, we needed `<sos>`,
`<eos>`, `<pad>`, and `<unk>` special tokens. Which of these do we need
for character-level generation, and why?

*Your answer:*

---

In [4]:
# Character vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [char_to_idx[c] for c in s]

def decode(ids):
    return ''.join(idx_to_char[i] for i in ids)

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

# Encode entire corpus
data = torch.tensor(encode(text), dtype=torch.long)

# Train/val split
split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]
print(f"\nTrain: {len(train_data):,} chars")
print(f"Val:   {len(val_data):,} chars")

Vocabulary size: 65
Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Train: 1,003,854 chars
Val:   111,540 chars


In [5]:
# Hyperparameters
CONTEXT_LEN = 256    # characters of context the model sees
BATCH_SIZE = 64

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


def get_batch(split):
    """
    Sample a random batch of context windows.

    Returns:
        x: (batch_size, context_len)     — input characters
        y: (batch_size, context_len)     — target characters (shifted by 1)
    """
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(data_split) - CONTEXT_LEN, (BATCH_SIZE,))
    x = torch.stack([data_split[i   : i + CONTEXT_LEN]     for i in ix])
    y = torch.stack([data_split[i+1 : i + CONTEXT_LEN + 1] for i in ix])
    return x.to(device), y.to(device)


# Verify
xb, yb = get_batch('train')
print(f"Input batch:  {xb.shape}")   # (64, 256)
print(f"Target batch: {yb.shape}")   # (64, 256)
print(f"\nInput:  '{decode(xb[0, :30].tolist())}'")
print(f"Target: '{decode(yb[0, :30].tolist())}'")
print("(Target is shifted by one character)")

Using device: cpu
Input batch:  torch.Size([64, 256])
Target batch: torch.Size([64, 256])

Input:  'vel out
My weaved-up folly? Ge'
Target: 'el out
My weaved-up folly? Gen'
(Target is shifted by one character)


---
## Section 3: Components from the NMT Notebook (PROVIDED)

These are copied directly from the Transformer NMT notebook.
You'll use them as building blocks. Read through them — you've seen
all of this before.

In [6]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Same function from our NMT notebook. No changes needed.

    Q: (batch, num_heads, seq_len_q, d_k)
    K: (batch, num_heads, seq_len_k, d_k)
    V: (batch, num_heads, seq_len_k, d_v)
    """
    d_k = K.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

In [7]:
class MultiHeadAttention(nn.Module):
    """
    Same class from our NMT notebook. No changes needed.

    Remember: this handles both self-attention and cross-attention
    depending on what you pass as query, key, value.
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, -1, self.d_model)

        output = self.W_o(attn_output)
        return output, attn_weights

In [8]:
class FeedForward(nn.Module):
    """
    Same class from our NMT notebook. No changes needed.
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

For reference, here is the **NMT decoder layer** from our translation notebook.
**Do not use this cell directly** — it's here so you can see what you're modifying
in the next section.

```python
class DecoderLayer(nn.Module):  # NMT VERSION — DO NOT COPY AS-IS
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)   # ← encoder-dependent
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)                        # ← for cross-attn
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)                       # ← for cross-attn
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask, tgt_mask):
        # 1. Masked self-attention
        attn_output, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_output))
        
        # 2. Cross-attention to encoder output        ← encoder-dependent
        attn_output, _ = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout2(attn_output))
        
        # 3. Feed-forward
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_output))
        
        return x
```

Look at the lines marked `← encoder-dependent`. Those exist because the NMT
decoder needs to look at the source sentence. In a GPT, there is no source
sentence.

---
## Section 4: YOUR TASK — GPT Decoder Layer

Modify the NMT `DecoderLayer` to create a `GPTBlock`.

**Questions to answer before you write code:**

**Q1:** The NMT decoder layer has three sublayers. Which one depends on the
encoder? What should you do with it?

*Your answer:*

**Q2:** The NMT decoder layer's `forward` method takes `encoder_output` and
`src_mask` as arguments. Does your GPTBlock need these? What arguments
does your forward method need?

*Your answer:*

**Q3:** How many LayerNorm modules does your GPTBlock need? How many did
the NMT decoder have?

*Your answer:*

---

Now write the code. Your `GPTBlock` should:
- Take `(d_model, num_heads, d_ff, dropout)` in `__init__`
- Have a `forward(self, x, mask)` method
- Use the `MultiHeadAttention` and `FeedForward` classes from Section 3
- Include residual connections and LayerNorm

In [9]:
class GPTBlock(nn.Module):
    """
    A single GPT decoder layer.
    This is the NMT DecoderLayer with ______ removed.

    Two sublayers:
        1. Masked self-attention + Add & Norm
        2. Feed-forward + Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # TODO: Define the layers you need
        # Hint: Look at the NMT DecoderLayer above.
        #       What do you keep? What do you remove?
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        # TODO: Implement the forward pass
        # Hint: Same structure as NMT DecoderLayer,
        #       minus the encoder-dependent parts.
        attn_output, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm2(x+self.dropout2(ff_output))
        return x

In [10]:
# ============================================================
# TEST CELL — Run this to verify your GPTBlock
# ============================================================

# Test with d_model=64, 4 heads, batch=2, seq_len=16
_test_block = GPTBlock(d_model=64, num_heads=4, d_ff=128, dropout=0.0)
_test_x = torch.randn(2, 16, 64)
_test_mask = torch.tril(torch.ones(16, 16)).unsqueeze(0).unsqueeze(0)

_test_out = _test_block(_test_x, _test_mask)

assert _test_out.shape == (2, 16, 64), \
    f"Expected shape (2, 16, 64), got {_test_out.shape}"

# Verify it doesn't expect encoder_output
import inspect
_sig = inspect.signature(_test_block.forward)
assert 'encoder_output' not in _sig.parameters, \
    "Your GPTBlock shouldn't take encoder_output — there's no encoder!"

print("GPTBlock tests passed!")
print(f"  Input shape:  {_test_x.shape}")
print(f"  Output shape: {_test_out.shape}")
print(f"  Parameters:   {sum(p.numel() for p in _test_block.parameters()):,}")

del _test_block, _test_x, _test_mask, _test_out

GPTBlock tests passed!
  Input shape:  torch.Size([2, 16, 64])
  Output shape: torch.Size([2, 16, 64])
  Parameters:   33,472


---
## Section 5: YOUR TASK — CharGPT Model

Now assemble the full model. For reference, here's what the NMT Transformer
looked like:

```
NMT:
  Source → src_embedding → positional_encoding → encoder_layers → encoder_output
  Target → tgt_embedding → positional_encoding → decoder_layers(encoder_output) → linear → logits
```

**Questions to answer before you write code:**

**Q1:** The NMT model has separate `src_embedding` and `tgt_embedding`.
Why did it need two? How many do you need?

*Your answer:*

**Q2:** The NMT model uses sinusoidal positional encoding (fixed, not learned).
For a fixed-length context window, a simpler option is a learned position
embedding — just `nn.Embedding(context_len, d_model)`. What's the tradeoff?

*Your answer:*

**Q3:** The NMT model has `make_src_mask` and `make_tgt_mask`. Which do you
need? Can you simplify the mask you keep? (Hint: think about whether your
input has padding.)

*Your answer:*

**Q4:** Draw (on paper or in your head) the data flow for your model:

```
Input characters → ??? → ??? → ??? → logits over character vocabulary
```

Fill this in before writing code.

*Your answer:*

---

Now write the model. It should:
- Take `(vocab_size, d_model, num_heads, num_layers, d_ff, context_len, dropout)` in `__init__`
- Have a `forward(self, idx, targets=None)` method that returns `(logits, loss)`
- Use your `GPTBlock` from Section 4
- Use `nn.Embedding` for both token and position embeddings

In [ ]:
class CharGPT(nn.Module):
    """
    Character-level GPT.

    Architecture:
        Token embedding + Position embedding
        → N × GPTBlock
        → LayerNorm
        → Linear projection to vocabulary
    """
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff,
                 context_len, dropout=0.1):
        super().__init__()
        self.context_len = context_len

        # TODO: Define your layers
        # You need:
        #   - Token embedding (characters → d_model vectors)
        #   - Position embedding (position index → d_model vectors)
        #   - Dropout
        #   - A stack of GPTBlocks
        #   - Final LayerNorm
        #   - Output projection (d_model → vocab_size)
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Embedding(context_len, d_model)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)

        self.layers = nn.ModuleList(
            [GPTBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.output_projection = nn.Linear(d_model, vocab_size)

        # Pre-compute the causal mask
        # This is the lower-triangular mask from our NMT notebook.
        # Question: why don't we need a padding mask here?
        mask = torch.tril(torch.ones(context_len, context_len))
        self.register_buffer('mask', mask.view(1, 1, context_len, context_len))

    def forward(self, idx, targets=None):
        """
        idx:     (B, T) — input character indices
        targets: (B, T) — target character indices (optional, for computing loss)

        Returns:
            logits: (B, T, vocab_size)
            loss:   scalar (if targets provided, else None)
        """
        B, T = idx.shape

        # TODO: Implement the forward pass
        #
        # Steps:
        #   1. Token embeddings: idx → (B, T, d_model)
        #   2. Position embeddings: need positions 0, 1, ..., T-1
        #   3. Add token + position embeddings, apply dropout
        #   4. Pass through GPTBlock stack
        #   5. Final LayerNorm
        #   6. Project to vocabulary logits
        #
        # For the mask, use:  self.mask[:, :, :T, :T]
        # (This slices the pre-computed mask to match sequence length T)
        #
        # For position embeddings, you need:
        #   torch.arange(T, device=idx.device)
        # to create position indices [0, 1, 2, ..., T-1]

        x_tok = self.embedding(idx)
        x_pos = self.pos_encoding(torch.arange(T, device=idx.device))
        x = x_tok + x_pos

        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, self.mask[:, :, :T, :T])

        x = self.layer_norm(x)

        logits = self.output_projection(x)

        # Compute loss if targets provided
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

In [18]:
# ============================================================
# TEST CELL — Run this to verify your CharGPT model
# ============================================================

# Small test model
_test_model = CharGPT(
    vocab_size=vocab_size,
    d_model=64,
    num_heads=4,
    num_layers=2,
    d_ff=128,
    context_len=32,
    dropout=0.0
).to(device)

# Test forward pass without targets
_test_idx = torch.randint(0, vocab_size, (2, 32), device=device)
_test_logits, _test_loss = _test_model(_test_idx)

assert _test_logits.shape == (2, 32, vocab_size), \
    f"Expected logits shape (2, 32, {vocab_size}), got {_test_logits.shape}"
assert _test_loss is None, \
    "Loss should be None when targets are not provided"

# Test forward pass with targets
_test_targets = torch.randint(0, vocab_size, (2, 32), device=device)
_test_logits, _test_loss = _test_model(_test_idx, _test_targets)

assert _test_loss is not None, \
    "Loss should not be None when targets are provided"
assert _test_loss.shape == (), \
    f"Loss should be a scalar, got shape {_test_loss.shape}"

# Loss should be close to -ln(1/vocab_size) for a random model
_expected = math.log(vocab_size)
assert abs(_test_loss.item() - _expected) < 1.0, \
    f"Initial loss {_test_loss.item():.2f} is too far from expected {_expected:.2f}"

# Test with shorter sequence (should work due to mask slicing)
_test_short = torch.randint(0, vocab_size, (2, 10), device=device)
_test_logits_short, _ = _test_model(_test_short)
assert _test_logits_short.shape == (2, 10, vocab_size), \
    f"Should handle shorter sequences. Expected (2, 10, {vocab_size}), got {_test_logits_short.shape}"

print("CharGPT tests passed!")
print(f"  Logits shape: {_test_logits.shape}")
print(f"  Initial loss: {_test_loss.item():.4f} (expected ~{_expected:.4f})")
print(f"  Parameters:   {sum(p.numel() for p in _test_model.parameters()):,}")

del _test_model, _test_idx, _test_targets, _test_logits, _test_loss

CharGPT tests passed!
  Logits shape: torch.Size([2, 32, 65])
  Initial loss: 4.2232 (expected ~4.1744)
  Parameters:   77,505


---
## Section 6: Instantiate the Real Model

Now create the actual model you'll train. These hyperparameters give
roughly 10-15M parameters — much larger than our char-RNN but small
enough to train in a reasonable time.

In [19]:
# Model hyperparameters
D_MODEL = 384
NUM_HEADS = 6       # d_k = 384/6 = 64 per head
NUM_LAYERS = 6
D_FF = 1536         # 4 × d_model
DROPOUT = 0.2

model = CharGPT(
    vocab_size=vocab_size,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    context_len=CONTEXT_LEN,
    dropout=DROPOUT
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"\nArchitecture: d_model={D_MODEL}, heads={NUM_HEADS}, "
      f"layers={NUM_LAYERS}, d_ff={D_FF}")
print(f"Context window: {CONTEXT_LEN} characters")
print(f"Vocabulary: {vocab_size} characters")

# Quick sanity check with real data
xb, yb = get_batch('train')
logits, loss = model(xb, yb)
print(f"\nSanity check:")
print(f"  Logits shape: {logits.shape}")
print(f"  Initial loss: {loss.item():.4f} (expected ~{math.log(vocab_size):.4f})")

Total parameters: 10,795,841

Architecture: d_model=384, heads=6, layers=6, d_ff=1536
Context window: 256 characters
Vocabulary: 65 characters

Sanity check:
  Logits shape: torch.Size([64, 256, 65])
  Initial loss: 4.3733 (expected ~4.1744)


---
## Section 7: Text Generation (PROVIDED)

This is the autoregressive generation loop — same idea as `greedy_translate`
from the NMT notebook, but with temperature scaling and top-k sampling
for more interesting output.

**Question:** In our NMT model, the generate loop ran the encoder once and
then called the decoder repeatedly. Here we just call `model()` repeatedly
with a growing input. Why don't we need a separate encode step?

*Your answer:*

---

In [20]:
@torch.no_grad()
def generate(model, prompt="ROMEO:\n", max_new_tokens=300,
             temperature=0.8, top_k=40):
    """
    Generate text from a prompt.

    temperature: >1 = more random, <1 = more deterministic
    top_k: only sample from the top k most likely characters
    """
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        # Crop to context window
        idx_cond = idx[:, -model.context_len:]

        # Forward pass
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        # Top-k filtering
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        # Sample
        probs = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)

    model.train()
    return decode(idx[0].tolist())


# Test with untrained model
print("Untrained model output (should be random characters):")
print(generate(model, max_new_tokens=100))

Untrained model output (should be random characters):
ROMEO:
AGVNQxVse.vzezjZgSrj3gjliZejlC;YQqA3FOBW&w-3SgpdvZ;OQZunQ,pQD,gC.RZGHV -EGZI.vGZ;TAWOzzWELHkcKT.xH-W


---
## Section 8: Training (PROVIDED)

The training loop is provided. It uses:
- AdamW optimizer with weight decay
- Cosine learning rate schedule with warmup
- Gradient clipping
- Periodic evaluation and sample generation

In [21]:
@torch.no_grad()
def estimate_loss(model, eval_iters=50):
    """Estimate train and val loss over several random batches."""
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        batch_losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            batch_losses.append(loss.item())
        losses[split] = np.mean(batch_losses)
    model.train()
    return losses

In [22]:
# Training configuration
MAX_ITERS = 5000
EVAL_INTERVAL = 250
SAMPLE_INTERVAL = 1000
LEARNING_RATE = 3e-4
MIN_LR = 3e-5
WARMUP_ITERS = 200

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)


def get_lr(step):
    """Cosine learning rate schedule with warmup."""
    if step < WARMUP_ITERS:
        return LEARNING_RATE * step / WARMUP_ITERS
    progress = (step - WARMUP_ITERS) / (MAX_ITERS - WARMUP_ITERS)
    return MIN_LR + 0.5 * (LEARNING_RATE - MIN_LR) * (1 + math.cos(math.pi * progress))

In [ ]:
# Training loop
train_losses = []
val_losses = []
eval_steps = []

print(f"Training for {MAX_ITERS:,} iterations...")
print(f"  {BATCH_SIZE} windows × {CONTEXT_LEN} chars = "
      f"{BATCH_SIZE * CONTEXT_LEN:,} chars/batch")
print()

start_time = time.time()

for step in range(MAX_ITERS):
    # Update learning rate
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Get batch and compute loss
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    # Backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # Evaluate
    if step % EVAL_INTERVAL == 0 or step == MAX_ITERS - 1:
        losses = estimate_loss(model)
        elapsed = time.time() - start_time
        print(f"Step {step:5d} | Train: {losses['train']:.4f} | "
              f"Val: {losses['val']:.4f} | LR: {lr:.2e} | {elapsed:.0f}s")
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        eval_steps.append(step)

    # Generate sample
    if step > 0 and step % SAMPLE_INTERVAL == 0:
        print(f"\n{'─' * 50}")
        print(f"Sample at step {step}:")
        print(f"{'─' * 50}")
        print(generate(model, max_new_tokens=200))
        print(f"{'─' * 50}\n")

print(f"\nDone in {(time.time()-start_time)/60:.1f} minutes")

Training for 5,000 iterations...
  64 windows × 256 chars = 16,384 chars/batch

Step     0 | Train: 4.3825 | Val: 4.3801 | LR: 0.00e+00 | 76s
Step   250 | Train: 2.4161 | Val: 2.4276 | LR: 3.00e-04 | 768s
Step   500 | Train: 1.9687 | Val: 2.0514 | LR: 2.97e-04 | 1446s
Step   750 | Train: 1.7178 | Val: 1.8790 | LR: 2.91e-04 | 2171s
Step  1000 | Train: 1.5802 | Val: 1.7803 | LR: 2.82e-04 | 4067s

──────────────────────────────────────────────────
Sample at step 1000:
──────────────────────────────────────────────────
ROMEO:
Ah, is are us counder of thing them speak of at heir with in.

FRIAR GAURENCE:
Come the rayal hing the here, and thou hear must the capped
Then of the reid in exparch as prot eyel
In than I but bout I
──────────────────────────────────────────────────

Step  1250 | Train: 1.4991 | Val: 1.7041 | LR: 2.69e-04 | 4697s
Step  1500 | Train: 1.4435 | Val: 1.6632 | LR: 2.54e-04 | 5339s
Step  1750 | Train: 1.3966 | Val: 1.6232 | LR: 2.36e-04 | 6009s
Step  2000 | Train: 1.3693 

---
## Section 9: Training Curves

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(eval_steps, train_losses, label='Train', marker='o', markersize=4)
plt.plot(eval_steps, val_losses, label='Val', marker='s', markersize=4)
plt.axhline(y=math.log(vocab_size), color='gray', linestyle='--',
            alpha=0.5, label=f'Random ({math.log(vocab_size):.2f})')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('CharGPT Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 10: Play with Your Model

In [ ]:
# Try different prompts
prompts = [
    "ROMEO:\n",
    "JULIET:\nO Romeo, ",
    "HAMLET:\nTo be, or not to be, ",
    "First Citizen:\n",
]

for prompt in prompts:
    print("=" * 50)
    print(f"Prompt: {repr(prompt)}")
    print("=" * 50)
    print(generate(model, prompt=prompt, max_new_tokens=300, temperature=0.8))
    print()

In [ ]:
# Compare temperatures
prompt = "KING HENRY:\nOnce more unto the breach, "

for temp in [0.3, 0.8, 1.2]:
    print(f"{'=' * 50}")
    print(f"Temperature = {temp}")
    print(f"{'=' * 50}")
    print(generate(model, prompt=prompt, max_new_tokens=200, temperature=temp))
    print()

---
## Section 11: Reflection Questions

Answer these after training your model.

**Q1:** Compare the output of your CharGPT to your char-RNN from our earlier
notebook. What specific improvements do you notice? What's still broken?

*Your answer:*

**Q2:** Look at your training curves. Is there a gap between train and val
loss? What does this tell you? What could you do about it?

*Your answer:*

**Q3:** In our NMT Transformer, the decoder had three sublayers. Your GPT
has two. The missing sublayer (cross-attention) connected the decoder to
the encoder. In a GPT, where does the model get its "source" information
from instead?

*Your answer:*

**Q4:** You used the same `MultiHeadAttention` and `FeedForward` classes
from the NMT notebook without modification. What does this tell you about
how general these components are?

*Your answer:*

**Q5:** GPT-4 uses essentially the same decoder-only architecture you just
built, but with ~1.8 trillion parameters instead of ~15 million, trained on
~13TB of text instead of 1MB. What do you think changes qualitatively
as you scale up by a factor of 100,000×?

*Your answer:*

**Q6:** Temperature controls the randomness of generation. At temperature=0.3,
the model is conservative and repetitive. At temperature=1.2, it's creative
but makes more mistakes. Why does this tradeoff exist? 

*Your answer:*